# 00 — TEMPUS Background: Building Blocks and Concepts
**What this notebook does:** Gives some overview of relativistic timescales and Introduces every moving part of the TEMPUS backend.

**Required packages to run this notebook:** `tempus` (installed), `calcephpy`, `astropy`, `spiceypy`

---
## 1  Relativistic Timescales


### 1.1  The IAU Coordinate Time Hierarchy

General relativity requires different time coordinates for different reference frames. The IAU resolutions define the following:

| Scale | Full name | Frame | Notes |
|---|---|---|---|
| **TAI** | International Atomic Time | Earth surface | Atomic clock average; no leap seconds |
| **TT** | Terrestrial Time | Earth geocentre | TT = TAI + 32.184 s; used for observations |
| **TDB** | Barycentric Dynamical Time | Solar system barycentre | Differs from TT by ≤ 2 ms; linear avg = TT |
| **TCB** | **Barycentric Coordinate Time** | Solar system barycentre | Primary relativistic scale; BCRS |
| **TCG** | Geocentric Coordinate Time | Earth geocentre | Co-moving with Earth; GCRS |
| **TCL** | Lunar-centric Coordinate Time | Moon centre | Co-moving with the Moon |
| **TCM** | Mars-centric Coordinate Time | Mars centre | Co-moving with Mars |
| **TCX** | Generic notation | Any body X | TEMPUS computes TCX–TCB for any X |

**TEMPUS computes TCX–TCB**: the accumulated relativistic offset between body X's proper coordinate time and the solar-system barycentric reference. This offset grows secularly (e.g. for Earth: ~1.28 ms/day) plus oscillates periodically with orbital motion.


### 1.2 Reference Epochs

Established TCX–TCB offsets (like for Earth) are set to **zero at 1977 January 1.0 TAI**, i.e. `JD 2443144.5003725 TT`. To follow this convention TEMPUS should be instructed to start integration at this epoch. 1977 was when the SI second was adopted as the international standard, and thus was choosen as a convenient zero-point for the drift between relativistic scales.

Another commonly referenced epoch is **J2000.0**, defined as `JD 2451545.0` in TT (noon of 1 January 2000) — ephemerides releases such as INPOP start their integration from it. The cell below expresses both reference epochs in every timescale.

In [1]:
from astropy.time import Time

DAY_S = 86400.0
JD_J2000 = 2451545.0

epochs = {
    "1977 origin (TCG-TCB=0 anchor)": Time(Time("1977-01-01T00:00:00", scale="tai"), format="jd", scale="tt"),
    "J2000.0 (JD 2451545.0 TT)":      Time(JD_J2000, format="jd", scale="tt"),
}

for label, t_tt in epochs.items():
    print(f"=== {label} ===")
    for name in [ "tai", "tt", "tdb", "tcb", "tcg"]: #you may also add "utc"
        jd = getattr(t_tt, name).jd
        print(f"  {name.upper():4s}: JD = {jd:.10f}   ({(jd - t_tt.tt.jd)*DAY_S:+.6f} s vs TT)")
    print()

=== 1977 origin (TCG-TCB=0 anchor) ===
  TAI : JD = 2443144.5000000000   (-32.184014 s vs TT)
  TT  : JD = 2443144.5003725002   (+0.000000 s vs TT)
  TDB : JD = 2443144.5003724992   (-0.000080 s vs TT)
  TCB : JD = 2443144.5003725002   (+0.000000 s vs TT)
  TCG : JD = 2443144.5003725002   (+0.000000 s vs TT)

=== J2000.0 (JD 2451545.0 TT) ===
  TAI : JD = 2451544.9996274998   (-32.184014 s vs TT)
  TT  : JD = 2451545.0000000000   (+0.000000 s vs TT)
  TDB : JD = 2451544.9999999991   (-0.000080 s vs TT)
  TCB : JD = 2451545.0001302510   (+11.253691 s vs TT)
  TCG : JD = 2451545.0000058548   (+0.505851 s vs TT)



In [2]:
# Same JD number, different scale ⇒ NOT the same physical instant:
j2000_as_tdb = Time(JD_J2000, format="jd", scale="tdb")
j2000_as_tcb = Time(JD_J2000, format="jd", scale="tcb")

print("JD 2451545.0 interpreted natively in each scale (≠ same instant):")
print(f"  TDB → TT offset: {(j2000_as_tdb.tt.jd - JD_J2000)*DAY_S:+.6f} s")
print(f"  TCB → TT offset: {(j2000_as_tcb.tt.jd - JD_J2000)*DAY_S:+.6f} s")

JD 2451545.0 interpreted natively in each scale (≠ same instant):
  TDB → TT offset: +0.000080 s
  TCB → TT offset: -11.253691 s


From these tables we can see, if we want to evaluate ephemerides across different timescales (like INPOP TDB and TCB releases), we have to be carefull which epochs and JD values we query them with. JD 2451545 might mean a different physical instant when you query a TDB vs a TCB ephemeris.  The act to make sure its the same instant we can call _epoch-correction_. 

Another aspect is the scaling of the time-steps, and the scaling of position and GM values/units (discussed below in 1.3), in order the be fully consistent across TDB and TCB ephemerides. 

### 1.3  TCB vs TDB — the L_B Scaling

TDB and TCB are related by a **constant linear drift**:
$$\mathrm{TDB} = \mathrm{TCB} - L_B\,(\mathrm{TCB} - T_0) + \mathrm{TDB}_0$$

where $L_B = 1.550519768 \times 10^{-8}$ (IAU 2006 resolution B3), $T_0$ is the 1977 epoch and $\mathrm{TDB}_0$ a small $-65.5\mu s$ offset (chosen so that TDB stays closer to TT).

**Practical consequence for TEMPUS:**
- Position Vectors in a TDB ephemeris are scaled by $(1 - L_B)$ relative to TCB (BCRS) based reference frames
- A step of 0.75 days in TDB ≠ 0.75 days in TCB -- TEMPUS has routines to correct for this if needed
- GM constants are similarly rescaled by $(1 - L_B)^3 / (1 - L_B)^2 = (1 - L_B)$


To compare INPOP's TCB and TDB releases at the same physical instant, TEMPUS provides `jd_tt_to_tdb()` / `jd_tt_to_tcb()` — they read the epoch correction directly off each ephemeris file's own Chebyshev series (CALCEPH targets 16/17), so the result is self-consistent with that file rather than going through astropy. The cell below uses them to align epochs, then checks the $L_B$ position-scaling formula above at a few points spanning a century.

In [3]:
from calcephpy import CalcephBin
from pathlib import Path
import numpy as np
from tempus import jd_split, jd_tt_to_tdb, jd_tt_to_tcb

tdb_path = "../data/ephem/inpop19a_TDB_m100_p100_tt.dat"
tcb_path = "../data/ephem/inpop19a_TCB_m100_p100_tcg.dat"
eph_tdb = CalcephBin.open(str(Path(tdb_path).resolve()))
eph_tcb = CalcephBin.open(str(Path(tcb_path).resolve()))

L_B  = 1.550519768e-8
AU_m = eph_tcb.getconstant("AU") * 1e3 #reading the IAU definition from the ephemeris, and converting to m

print(f"{'years from J2000':>18}  {'uncorrected [m]':>16}  {'L_B-corrected [m]':>18}")
for yr in (0, 50, 100):
    tt_int, tt_frac = jd_split(JD_J2000 + yr * 365.25)
    tdb_int, tdb_frac = jd_tt_to_tdb(tt_int, tt_frac, tdb_path)
    tcb_int, tcb_frac = jd_tt_to_tcb(tt_int, tt_frac, tcb_path)

    # target "3" is the Earth-Moon barycenter, and "0" relative to the solar system barycenter (SSB)
    pv_tdb = np.array(eph_tdb.compute(tdb_int, tdb_frac, 3, 0))  
    pv_tcb = np.array(eph_tcb.compute(tcb_int, tcb_frac, 3, 0))

    raw  = np.linalg.norm(pv_tcb[:3]             - pv_tdb[:3]) * AU_m
    corr = np.linalg.norm(pv_tcb[:3] * (1 - L_B) - pv_tdb[:3]) * AU_m
    print(f"{yr:>18}  {raw:>16.2f}  {corr:>18.6f}")

  years from J2000   uncorrected [m]   L_B-corrected [m]
                 0           2277.54            0.000222
                50           2272.91            0.000249
               100           2280.09            0.000215


Without the $L_B$ rescaling, the TCB/TDB position mismatch is on the order of kilometers; with it, the residual stays sub-millimeter throughout — confirming the scaling from §1.3 is correct and required.

### 1.4  SPICE ET and JD conventions

SPICE uses **ET** (Ephemeris Time) = seconds past J2000.0 in TDB. TEMPUS works in **split JD** (integer + fractional part) for numerical precision. Useful reference points:
- `ET = 0` ↔ `JD 2451545.0 TDB` ≈ 2000-01-01 12:00 TDB
- `JD_J2000 = 2451545.0` (TT by IAU definition; differs from TDB by < 2 ms)
- TEMPUS context: in the `.csv` outputs we also provide (besides the split JDs) an epoch column in terms of seconds past J2000 (similar to ET used in SPICE).


In [4]:

import spiceypy as spice
spice.furnsh("../data/kernels/naif0012.tls")
print(spice.et2utc(0.0, "ISOC", 3)) #32 accumulated leap seconds + 32.184 s (TT-TAI) offset from 1200UTC

2000-01-01T11:58:55.816


Takeaway for INPOP / SPICE usage:
- SPICE ET=0  means JD 2451545.0 TDB  (≈ J2000 TT, differs by < 2 ms)
- INPOP TCB kernels: J2000 TT instant ≈ JD 2451545.0 + 11.25 s in TCB

### 1.5  Where this shows up in TEMPUS

- **Split-JD precision (1.1)** — `TimeScalar` / `jd_split()` in `model.py` keep every epoch as int+frac JD, avoiding float64 cancellation.
- **1977 reference epoch (1.2)** — hardcoded as `_T0_INT` / `_T0_FRAC` in `model.py`, anchoring the TT–TCG drift term in `jd_tt_to_tcb()`.
- **TT→TDB/TCB conversion & $L_B$ scaling (1.3)** — `jd_tt_to_tdb()` / `jd_tt_to_tcb()` read CALCEPH targets 16/17 off the ephemeris file; the same `L_B` rescales positions/GM in `_build_context()` and corrects the integration rate in `Delta_TCX` whenever `eph.timescale() == "TDB"`.
- **GM unit ambiguity** — `CalcephEphem.detect_gm_unit()` / `gm_m3_s2()` (Section 3) generalize the Sun-GM check done by hand there.
- **ET seconds past J2000 (1.4)** — `jd_parts_to_et()` is the integration variable inside `Delta_TCX`.

---
## 2  Ephemeris Files


### 2.1  What an Ephemeris File Contains

A planetary ephemeris stores pre-computed positions and velocities of solar-system bodies as Chebyshev polynomial coefficients on fixed time granules. TEMPUS reads them via [CALCEPH](https://calceph.imcce.fr/).

| Format | Extension | Producer | Notes |
|---|---|---|---|
| INPOP binary | `.dat` | IMCCE/OCA | Native CALCEPH format; can carry internal TCG–TCB |
| INPOP Chebyshev | `.tcheb` | IMCCE/OCA | Compact binary; same API |
| SPICE BSP | `.bsp` + `.tpc` | JPL/NASA | Needs leap-second kernel (`.tls`) |
| EPM | `.bsp` + `.tpc` | IAA RAS | Same SPICE format as DE |

**TDB vs TCB flavours:** INPOP releases come in both. DE series (DE430, DE440) and EPM are TDB only. TEMPUS auto-detects the timescale from the file and applies $L_B$ rescaling for TDB files, in order to use the relativistic time formulas in TCB compatible units.


### 2.2  Downloading Ephemeris Files

Place CALCEPH and INPOP related files under `../data/ephem/` and SPICE kernels under `../data/kernels/`. The commands below are also available as a runnable script: `scripts/download_data.sh [group ...]` (safe to re-run; skips files already present).

In [5]:
# ── INPOP (both TDB and TCB flavours, native CALCEPH .dat format) ───────────
# INPOP19a and INPOP21a are available from the IMCCE FTP:
# wget https://ftp.imcce.fr/pub/ephem/planets/inpop19a/inpop19a_TCB_m100_p100_tcg.dat
# wget https://ftp.imcce.fr/pub/ephem/planets/inpop19a/inpop19a_TDB_m100_p100_tt.dat
# wget https://ftp.imcce.fr/pub/ephem/planets/inpop21a/inpop21a_TCB_m100_p100_tcg.dat
# wget https://ftp.imcce.fr/pub/ephem/planets/inpop21a/inpop21a_TDB_m100_p100_tt.dat
# Files named: inpop21a_*_m1000_p1000_*.dat  (m1000=minus 1000yr, p1000=plus 1000yr)

# ── JPL DE series (TDB, SPICE format) ──────────────────────────────────────
# DE440 planetary ephemeris (~110 MB)
# wget https://ssd.jpl.nasa.gov/ftp/eph/planets/bsp/de440.bsp
# DE440 gravitational parameters (required for GM lookups with SPICE)
# wget https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de440.tpc
# DE440 main-belt asteroid trajectories (~1 GB, 343 asteroids + 30 TNOs)
# wget https://ssd.jpl.nasa.gov/ftp/eph/small_bodies/asteroids_de441/sb441-n373s.bsp

# Leap-second kernel (required for SPICE time conversions), currently not used with TEMPUS
# wget https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls

# DE430 planetary ephemeris (older, no KBOs, but better for Moon ephemerides)
# wget https://ssd.jpl.nasa.gov/ftp/eph/planets/bsp/de430.bsp
# DE431 gravitational parameters (required for GM lookups with SPICE)
# wget https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de431.tpc
# DE430 main-belt asteroid trajectories (343 asteroids)
# wget https://ssd.jpl.nasa.gov/ftp/eph/small_bodies/asteroids_de430/ast343de430.bsp

# ── EPM2021 (TDB, SPICE format) ─────────────────────────────────────────────
# Available from IAA RAS: https://iaaras.ru/en/dept/ephemeris/epm/2021/
# wget https://ftp.iaaras.ru/pub/epm/EPM2021/SPICE/epm2021.bsp
# wget https://ftp.iaaras.ru/pub/epm/EPM2021/SPICE/epm2021.tpc


### 2.3  Header / GM-Supplement Files

Binary ephemeris files don't always carry every body's GM — INPOP `.dat` headers are missing some asteroid masses (§3.1), and DE/EPM SPICE kernels split GMs into a separate `.tpc`. TEMPUS fills these gaps via `--asteroid-gm-header`, which reads a JPL-style ASCII "header" file: `GROUP 1040` lists constant names, `GROUP 1041` lists the matching values (Fortran `D`-exponent doubles). `parse_de_header()` in `ephem.py` parses this format; `CalcephEphem.supplement_gm_from_header()` merges the values into the ephemeris' constant cache, with user-supplied values taking priority on collision.

The header files under `data/` fall into two distinct categories — worth knowing apart, since only one kind is fetchable by official URL:

| File | Category | Origin |
|---|---|---|
| `kernels/header.440` | official | JPL DE440 ascii header — full `KSIZE=2036 NCOEFF=1018` block with all planetary constants, not just GMs |
| `kernels/header.430_572` | official | JPL DE430 ascii header (`_572` is literally part of the server filename, not a version tag TEMPUS assigns) |
| `kernels/header.21h` | official | IAA RAS EPM2021H ascii header, same full-constants format |
| `kernels/header_kbo.440` | generated | `KSIZE=0 NCOEFF=0`, `GROUP 1040/1041` only — asteroid/TNO GMs only. Built by `scripts/header_supplement_from_twocol.py` from a two-column `id  GM` table found in a PDF release|
| `ephem/header_mba.19a`, `ephem/header_mba.21a` | generated | Same `KSIZE=0` format. Built by `scripts/header_supplement_file_generation.py` from INPOP `GM_...` lines |

The `KSIZE=0 NCOEFF=0` marker is the tell: official releases always carry the full ephemeris metadata block, while the locally-generated files only ever add the constants that were missing. The generated files are already committed to the repo — `scripts/header_supplement_*.py` only need rerunning for a new ephemeris release with different missing GMs. See `scripts/README.md` for details.


---
## 3  Building Block: `CalcephEphem` — Loading and Inspecting an Ephemeris


`CalcephEphem` is TEMPUS's ephemeris wrapper. It opens `.dat`, `.tcheb`, `.bsp` files (plus optional `.tpc` and `.tls` SPICE kernels) via a unified API.

On open it prints: file version and timescale enum `1=TDB`, `2=TCB`. The Sun GM is a useful sanity check — TEMPUS converts everything to SI (m³/s²) interally.


In [6]:
from tempus import CalcephEphem, jd_grid, Delta_TCX

eph = CalcephEphem.open("../data/ephem/inpop19a_TCB_m100_p100_tcg.dat")

# get version
print(eph.fileversion())
ts_label = eph.timescale()
print(f"CALCEPH timescale enum: {ts_label}")

INPOP19A
CALCEPH timescale enum: TCB


### 3.1  Reading Constants and Detecting GM Units

INPOP stores GM values in AU³/day² (stored as a fraction relative to the Sun). `detect_gm_unit()` checks the Sun GM against known AU³/day² and m³/s² values to determine the unit, then `gm_m3_s2(naif_id)` always returns SI regardless of source.


In [7]:
c = eph.constants()

# Try common sun keys
for k in ["GM_SUN", "GMS", "BODY10_GM", "GM10"]:
    if k in c:
        print("Raw Sun GM key:", k, "value:", c[k])

# If GM_SUN ~ 2.959e-4 -> AU^3/day^2; 
#        if ~ 1.327e11 -> km^3/s^2
print(f"unit mode: {eph.detect_gm_unit()}")

Raw Sun GM key: GM_SUN value: 0.00029591221287397545
unit mode: AU3/DAY2


In [8]:
# Optional: sanity check Sun GM to confirm unit conversion is correct
# NAIF key 10 is the Sun, this should return ~1.327124x10^20 m^3/s^2
sun_gm = eph.gm_m3_s2(10) 
print(f"Sun GM (SI): {sun_gm:.6e} m^3/s^2")

Sun GM (SI): 1.327124e+20 m^3/s^2


In [9]:
#print the keys in c = eph.constants()
for name in c.keys():
    print(name, c[name])

KSIZER 1226.0
VERSIO 19.1
FVERSI 0.0
FORMAT 11.0
UNITE 1.0
TIMESC 1.0
REDATE 2019.1213
CLIGHT 299792.458
GM_MER 4.9124805265337416e-11
GM_VEN 7.243452444955282e-10
GM_EMB 8.99701153352167e-10
GM_MAR 9.549548843618412e-11
GM_JUP 2.8253458846414165e-07
GM_SAT 8.459706204414447e-08
GM_URA 1.2920248458160606e-08
GM_NEP 1.5243573715205723e-08
GM_PLU 2.1668073524057028e-12
GM_SUN 0.00029591221287397545
JDEPOC 2451545.000130252
GAMMA 1.0
BETA 1.0
RSUN 696000.0107916178
RMOON 1738.000026948034
REARTH 6378.13669889427
K2E0 0.335
K2E1 0.32
K2E2 0.282
TAUE0 0.0
TAUE1 0.0
TAU22 0.0
TAUR0 0.0
TAUR1 0.007979387594672007
TAUR2 0.0028231876277091156
TAUO0 0.07800000120940544
TAUO1 -0.04400000068222871
TAUO2 -0.11300000175208737
J2ESEC -2.599999959686486e-09
OMEGAE 6.300387262311249
CMR2E 0.3306947357075919
SINJE 0.0
CMR2M 0.393140294559018
K2M 0.023559
TAUM 0.09433233368966375
CMR2CM 0.000275
KCMB 6.443479383181008e-09
J2SUN 2.0096262034991606e-07
J4SUN 0.0
ALPSUN 286.13
DELSUN 63.87
GM_RIN 0.0
RRING 

In this constants object read by CALCEPH from the binary .dat file, not all Asteroid Masses/GMs are listed. Why that is, is not understood yet. In the meantime we provide header_mba.19a and header_mba.21a for use with TEMPUS, which includes all MBA masses that were used for INPOP.

In [10]:
# close when done (or use as a context manager)
eph.close()

---
## 4  Building Block: Body parsing

TEMPUS identifies bodies by **NAIF IDs** (the NASA/JPL numbering standard). `parse_body` accepts a INPOP or SPICE style name string, a NAIF integer, or a string integer and returns a `BodyRef` object type.

| Body | NAIF ID | TCX name |
|---|---|---|
| Solar System Barycentre | 0 | TCB |
| Sun | 10 | — |
| Earth barycentre | 3 | — |
| Earth centre | 399 | TCG |
| Moon | 301 | TCL |
| Mars barycentre | 4 | TCM |
| Asteroid (e.g. Ceres) | 2000001 | — |

`ext_list(target)` returns the default list of perturbing bodies for a given target (Sun + all planets + Pluto, excluding the target itself) — what the CLI uses for `--ext auto`.


In [11]:
from tempus.bodies import parse_body, ext_list

target = parse_body("MARS BARYCENTER")  # or 4 or "4"
center = parse_body("SSB")              # or 0
ext = ext_list(target)                  # list[BodyRef], target excluded

print(f"For the Target {target.key}, the automatically resolved list of external bodies is:")
for b in ext:
    print(b)

For the Target MARS_BARYCENTER, the automatically resolved list of external bodies is:
BodyRef(naif=10, key='SUN', spice='SUN')
BodyRef(naif=1, key='MERCURY_BARYCENTER', spice='MERCURY BARYCENTER')
BodyRef(naif=2, key='VENUS_BARYCENTER', spice='VENUS BARYCENTER')
BodyRef(naif=3, key='EARTH_MOON_BARYCENTER', spice='EARTH MOON BARYCENTER')
BodyRef(naif=5, key='JUPITER_BARYCENTER', spice='JUPITER BARYCENTER')
BodyRef(naif=6, key='SATURN_BARYCENTER', spice='SATURN BARYCENTER')
BodyRef(naif=7, key='URANUS_BARYCENTER', spice='URANUS BARYCENTER')
BodyRef(naif=8, key='NEPTUNE_BARYCENTER', spice='NEPTUNE BARYCENTER')
BodyRef(naif=9, key='PLUTO_BARYCENTER', spice='PLUTO BARYCENTER')


---
## 5  Building Block: Asteroid Files


TEMPUS was tested with two asteroid trajectory files:

| File | Format | Bodies | Time range | Notes |
|---|---|---|---|---|
| `sb441-n373s.bsp` | SPICE BSP | 343 MBAs + 30 TNOs | ±100 yr | JPL DE441; needs `gm_de440.tpc` for GMs and additional header files for the KBOs GMs|
| `sim20201224Tyref13quar.dat` | INPOP binary | ~427 trajectories | ~2000–2024 | INPOP sim file; GMs inside the main `.dat` |

**GM availability:** not all bodies in the trajectory file have mass constants. Bodies without GM are silently skipped. Use `asteroid_indices_with_gm_in_main_file()` to see which are usable.


In [12]:
from calcephpy import CalcephBin

kernel_path = ["../data/kernels/sb441-n373s.bsp"]
#kernel_path = "../data/ephem/sim20201224Tyref13quar.dat"
eph = CalcephBin.open(kernel_path)

n = eph.getconstantcount()
m = eph.getpositionrecordcount()
print("number of constants:", n)
print("number of positions:", m)

for i in range(1, n + 1):   # CALCEPH uses 1-based indexing here
    name, value = eph.getconstantindex(i)
    print(i, name, value)

eph.close()

number of constants: 0
number of positions: 374


In [13]:
from tempus import CalcephEphem
#"../data/ephem/sim20201224Tyref13quar.dat"
#"../data/kernels/sb441-n373s.bsp"
eph_a = CalcephEphem.open(["../data/kernels/de440.bsp", "../data/kernels/gm_de440.tpc"], "../data/kernels/sb441-n373s.bsp")

In [14]:
eph_a.asteroid_indices_with_gm_in_main_file()

[1,
 2,
 3,
 4,
 7,
 10,
 15,
 16,
 31,
 52,
 65,
 87,
 88,
 107,
 433,
 511,
 704,
 18050000,
 18065803,
 18090482,
 18120347,
 18136108,
 18136199,
 18469705,
 18612095,
 18612687,
 51031823,
 51092511,
 118050000,
 118065803,
 118090482,
 118120347,
 118136108,
 118136199,
 118469705,
 118612095,
 118612687,
 151031823,
 151092511,
 218136108,
 918050000,
 918065803,
 918090482,
 918120347,
 918136108,
 918136199,
 918469705,
 918612095,
 918612687,
 951031823,
 951092511]

In [15]:
ast_id_list = eph_a.asteroid_object_ids_in_file()
for id in ast_id_list:
    print(id, end=" ")
print(f"\n {len(ast_id_list)} asteroids in the file")

2000001 2000002 2000003 2000004 2000005 2000006 2000007 2000008 2000009 2000010 2000011 2000012 2000013 2000014 2000015 2000016 2000017 2000018 2000019 2000020 2000021 2000022 2000023 2000024 2000025 2000026 2000027 2000028 2000029 2000030 2000031 2000032 2000034 2000035 2000036 2000037 2000038 2000039 2000040 2000041 2000042 2000043 2000044 2000045 2000046 2000047 2000048 2000049 2000050 2000051 2000052 2000053 2000054 2000056 2000057 2000058 2000059 2000060 2000062 2000063 2000065 2000068 2000069 2000070 2000071 2000072 2000074 2000075 2000076 2000077 2000078 2000079 2000080 2000081 2000082 2000083 2000084 2000085 2000086 2000087 2000088 2000089 2000090 2000091 2000092 2000093 2000094 2000095 2000096 2000097 2000098 2000099 2000100 2000102 2000103 2000104 2000105 2000106 2000107 2000109 2000110 2000111 2000112 2000113 2000114 2000115 2000117 2000118 2000120 2000121 2000124 2000127 2000128 2000129 2000130 2000132 2000134 2000135 2000137 2000139 2000140 2000141 2000143 2000144 2000145 

The `2000001`-style IDs are NAIF SPK asteroid IDs (`2000000 + asteroid_number`). TEMPUS normalises these internally so you can use either `1` (Ceres index) or `2000001` (NAIF ID) in `--asteroid-ids`.


In [16]:
eph_a.close()

---
## 6  Building Block: Split JD and Time Grid


A standard `float64` JD (~2451545.0) has only ~8 significant decimal places, giving a time resolution of only ~40 μs; far above sub-nanosecond accuracy.

TEMPUS represents every epoch as **integer + fractional** JD parts:
```
jd_int  = 2451545   # integer days (exact)
jd_frac = 0.000130  # fractional day (full float64 precision ~10 ps)
```
`jd_split(jd)` splits a float into a `TimeScalar(int_part, frac_part)` object. 

`jd_grid(start_int, start_frac, end_int, end_frac, step)` produces the integration grid.


In [17]:
from tempus import jd_split, TimeScalar
from tempus.model import jd_grid

J1977_TT = 2443144.5003725
J2000_TT = 2451545.0

start = jd_split(J1977_TT)
end   = jd_split(J2000_TT + 365.25 * 50)  # 50 years after J2000
print(f'Start: int={start.int_part}, frac={start.frac_part:.10f}')
print(f'End:   int={end.int_part},   frac={end.frac_part:.10f}')

grid = jd_grid(
    jd_start_int=start.int_part, jd_start_frac=start.frac_part,
    jd_end_int=end.int_part,     jd_end_frac=end.frac_part,
    jd_step=0.75,                # 18 hours - standard step size
)
print(f'Grid: {len(grid.jd_int)} steps, first JD = {grid.jd_int[0]+grid.jd_frac[0]:.6f}')

Start: int=2443144, frac=0.5003725002
End:   int=2469807,   frac=0.5000000000
Grid: 35551 steps, first JD = 2443144.500373


---
## 7  CLI Walkthrough: `tempus integrate`


The CLI is a thin wrapper around `Delta_TCX` (Section 8). Every CLI argument maps directly to a Python API parameter.

```
tempus integrate [required args] [optional args]
```


### 7.1  Ephemeris Input: `--eph-file`

```bash
--eph-file path/to/inpop19a_TCB_m100_p100_tcg.dat
```

Accepts one or more files. For SPICE: pass both the `.bsp` and the `.tpc` (GM file):
```bash
--eph-file de440.bsp gm_de440.tpc
```
TEMPUS auto-detects TDB or TCB from the file header.


### 7.2  Time Grid: `--jd-start`, `--jd-end`, `--jd-step`

```bash
--jd-start 2443144 0.5   # ~1977 epoch (int and frac)
--jd-end   2469807 0.5   # ~2027
--jd-step  0.75          # 18 hours steps
```

These JD values are interpreted in the native timescale of the loaded ephemerids (typtically TDB, or here TCB in case of INPOP TCB releases). Use `jd_tt_to_tdb()`/`jd_tt_to_tcb()` from `tempus` to compute the correct split-JD start epoch, if you want to provide exact epochs from TT JD values; see `01_validation_tcg_klioner.ipynb` for a running example.


### 7.3  Body Selection: `--target`, `--center`, `--ext`

```bash
--target MARS_BARYCENTER   # or: 4
--center SSB               # Solar System Barycentre (always for TCX–TCB)
--ext auto                 # all planets + Sun, excluding target (recommended)
```

Special targets for Earth: use `EARTH_CENTER` (NAIF 399, → TCG) not `EARTH_BARYCENTER` (NAIF 3).
For Moon: `MOON` (NAIF 301, → TCL).


### 7.4  Asteroid Perturbers: `--asteroid-file`, `--asteroid-ids`, `--asteroid-gm-header`

```bash
# Use all asteroids with GM from the INPOP sim file:
--asteroid-file ../data/ephem/sim20201224Tyref13quar.dat
--asteroid-gm-header ../data/ephem/header_mba.19a

# Use DE441 asteroid BSP + supplement GMs from a header file:
--asteroid-file ../data/kernels/sb441-n373s.bsp
--asteroid-gm-header ../data/kernels/header_kbo.440

# Select specific asteroids by index:
--asteroid-ids 1,2,3,4,10    # Ceres, Pallas, Juno, Vesta, Hygiea
```

If an asteroid has no GM constant it is silently skipped. The printed summary line shows how many were included.


### 7.5  Integration Options: `--integrator`, `--pn1`, `--longdouble`

```bash
--integrator am12    # Adams-Moulton 12-step (default, recommended)
                     # other options: trapz, rk4, am4, am8, am10, amk
--pn1                # use only O(c⁻²); omit O(c⁻⁴) correction terms
--longdouble         # experimental: 80-bit float for the integration, not currently recommended
```

AM12 is chosen because the integrand is smooth between ephemeris nodes; high-order multistep methods converge much faster than RK4 for this problem.


### 7.6  Output Columns: `--out`, `--out-rate`, `--out-tcg-tcb`, etc.

```bash
--out path/to/output.csv       # optional filepath/name; otherwise its automatic. columns always written: JD_int, JD_frac, SecondsPastJ2000, TCB_minus_TCX
--out-rate                     # add dTCX/dTCB rate column
--out-tcg-tcb                  # add TCG_minus_TCB from ephemeris (INPOP TCB only, used for validation during development)
--out-tt-tdb                   # add TT_minus_TDB from ephemeris (INPOP TDB only, used for validation during development)
```

A `.json` metadata sidecar is written alongside every CSV.


### 7.7  Full Example Command


In [18]:
# Full CLI example simple TCG (Earth) over 24 years 
import subprocess
from pathlib import Path
from tempus import jd_split, jd_tt_to_tcb

J2000_TT = 2451545.0

eph_file = '../data/ephem/inpop19a_TCB_m100_p100_tcg.dat'
out_csv  = '../output/nb00_earth_p24_INPOP19aTCB.csv'

s_int, s_frac = jd_tt_to_tcb(*jd_split(J2000_TT), eph_file)
e_int = s_int + 365.25 * 24   # 24 yr in TCB
e_frac = s_frac

cmd = [
    'tempus', 'integrate',
    '--eph-file',      str(eph_file),
    '--jd-start',      str(int(s_int)), str(s_frac),
    '--jd-end',        str(int(e_int)), str(e_frac),
    '--jd-step',       '0.75',
    '--target',        'EARTH_CENTER',
    '--out',           str(out_csv),
]
print(' '.join(cmd))
# Uncomment to actually run:
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout, res.stderr)

tempus integrate --eph-file ../data/ephem/inpop19a_TCB_m100_p100_tcg.dat --jd-start 2451545 0.00013025101812747615 --jd-end 2460311 0.00013025101812747615 --jd-step 0.75 --target EARTH_CENTER --out ../output/nb00_earth_p24_INPOP19aTCB.csv
Ephemeris version: INPOP19A
CALCEPH timescale: 2 (TCB)
Approx. mean drift (TCB-TCX): 1.279431 ms/day
Wrote ../output/nb00_earth_p24_INPOP19aTCB.csv
 


Or here a bash shell script example:

In [ ]:
EPH_FILES=("../data/kernels/de440.bsp" "../data/kernels/gm_de440.tpc")
AST_FILE="../data/kernels/sb441-n373s.bsp"
AST_GM_FILE="../data/kernels/header.440"
OUT_CSV="../output/nb00_moon_m100_p100_DE440_ast_BIG5.csv"

# JD start/end as single floats, in timescale of the input ephemeris (TDB for DE440)
# Here we use the timespan of 200 years centered on ~J2000, with 0.75d integration steps
JD_START_INT=$(( 2451545 - 36525 ))
JD_START_FRAC=0.0
JD_END_INT=$(( 2451545 + 36525 ))
JD_END_FRAC=0.0
JD_STEP=0.75

# --zero: TCB-TCL = 0 at the IAU 1977 origin (here on DE440's TDB axis) instead of at the first sample
tempus integrate \
    --eph-file "${EPH_FILES[@]}" \
    --asteroid-file "$AST_FILE" \
    --asteroid-gm-header "$AST_GM_FILE" \
    --asteroid-ids "1,2,4,7,324" \
    --jd-start "$JD_START_INT" "$JD_START_FRAC" \
    --jd-end   "$JD_END_INT"   "$JD_END_FRAC" \
    --jd-step  "$JD_STEP" \
    --target MOON \
    --center SSB \
    --ext auto \
    --integrator am12 \
    --out "$OUT_CSV" \
    --zero

---
## 8  Python API: Running an Integration with `Delta_TCX`


`Delta_TCX` is the core function the CLI calls internally. You may use it directly when you need to embed TEMPUS in a larger Python workflow.


In [20]:
from pathlib import Path
import numpy as np
from tempus import CalcephEphem, jd_split, jd_tt_to_tcb
from tempus.model import Delta_TCX, jd_grid
from tempus.bodies import parse_body, ext_list

J1977_TT = 2443144.5003725

eph_file = Path('../data/ephem/inpop19a_TCB_m100_p100_tcg.dat').resolve()

eph = CalcephEphem.open(eph_file)

s_int, s_frac = jd_tt_to_tcb(*jd_split(J1977_TT), eph_file)

target = parse_body('EARTH_CENTER')  # → TCG
center = parse_body('SSB')
ext    = ext_list(target)            # all planets + Sun except Earth

# 100 years fron J1977_TT
deltajd = 36525
grid = jd_grid(
    jd_start_int=int(s_int), jd_start_frac=np.float64(s_frac),
    jd_end_int=int(s_int) + deltajd,  jd_end_frac=np.float64(s_frac),
    jd_step=0.75,
)

delta, rate = Delta_TCX(
    eph=eph,
    jd_int=grid.jd_int,  jd_frac=grid.jd_frac,
    target=target.naif,  center=center.naif,
    ext_bodies=[b.naif for b in ext],
    gm_provider=lambda b: eph.gm_m3_s2(int(b)),
    integrator='am12',
    include_o4=True,     # include PN2 correction
    include_sun_j2=False,
)

print(f'TCB-TCG at end: {delta[-1]:.6f} s')
print(f'Mean drift: {(delta[-1]-delta[0])/(deltajd)*1e3:.4f} ms/day')

eph.close()

TCB-TCG at end: 46.731343 s
Mean drift: 1.2794 ms/day


---
## 9  `nb_utils` Wrapper: `run_tempus_integration`

The validation notebooks use `run_tempus_integration` from `nb_utils.py` instead of building the bash CLI command by hand. It takes `TimeScalar` start/end objects and forwards any extra keyword arguments as CLI flags:

- `kwarg=True` → bare flag (`out_rate=True` → `--out-rate`)
- `kwarg='value'` → `--kwarg-name value` (underscores become hyphens)
- `kwarg=False` or `kwarg=None` → flag omitted

It builds and runs the CLI and returns the `subprocess.CompletedProcess` result.


In [24]:
from pathlib import Path
from tempus import jd_split, jd_tt_to_tcb, TimeScalar
from nb_utils import run_tempus_integration

J1977_TT = 2443144.5003725
J2000_TT = 2451545.0
eph_file = Path('../data/ephem/inpop19a_TCB_m100_p100_tcg.dat').resolve()
out_csv  = Path('../output/nb00_earth_p100_INPOP19aTCB.csv').resolve()

# Build TimeScalar start/end — the wrapper calls .cli_args() to get [int, frac] strings
s_int, s_frac = jd_tt_to_tcb(*jd_split(J2000_TT), eph_file)
start = TimeScalar(int(s_int), s_frac)
end   = start + 365.25 * 100  # 100 years

# kwargs map 1-to-1 to CLI flags:
#   target='EARTH_CENTER'  → --target EARTH_CENTER
#   asteroid_file=...      → --asteroid-file ...
#   out_rate=True          → --out-rate
#   pn1=False              → (omitted, so PN2 is used by default)
res = run_tempus_integration(
    eph_file=eph_file,
    start=start,
    end=end,
    jd_step=0.75,
    target='EARTH_CENTER',
    out=out_csv,
    out_rate=True,
)

# res.returncode == 0 on success; output CSV is at out_csv
print(res.returncode)

Ephemeris version: INPOP19A
CALCEPH timescale: 2 (TCB)
Approx. mean drift (TCB-TCX): 1.279435 ms/day
Wrote /home/yan/PythonWorkdir/CNRS_Release/TEMPUS/output/nb00_earth_p100_INPOP19aTCB.csv
0
